# Adding user history to the recommendation engine

For many recommendation cases, such as film or purchase recommendation,
basing the next recommendation on one data point will be insufficient.
In these cases, you'll need to embed all or some of the user's history for more accurate
and relevant recommendations.

In this exercise, you'll extend your product recommendation system
to consider all of the products the user has previously visited,
which are stored in a list of dictionaries called user_history.

做一个基于 Embedding 的商品推荐系统（Recommendation System）。

核心思想非常简单：

根据用户过去喜欢/浏览过的商品，计算用户的“兴趣向量”，然后从其他商品中找出语义上最接近的商品。

你这段代码特别适合用来理解 Embedding + Vector Similarity + Recommendation。

流程：
```text
用户历史商品
    ↓
转换成文本
    ↓
生成 Embedding
    ↓
计算平均 Embedding
    ↓
用户兴趣向量
        ↓
与所有其他商品比较
        ↓
找出最相似的商品
        ↓
    推荐商品
```

Embedding 的关键不是每个数字本身有什么意义。

而是：

语义相似的文本，它们的向量通常会比较接近。

In [3]:
import numpy as np  # NumPy 用来进行数学计算
from all_functions import create_product_text, find_n_closest
from DATA import products
from openai_helper import create_embeddings

user_history = [
    {
        "title": "Remote-Controlled Dinosaur Toy",
        "short_description": "Roar into action with this remote-controlled dinosaur toy with lifelike movements.",
        "price": 49.99,
        "category": "Toys",
        "features": [
            "Realistic dinosaur sound effects",
            "Walks and roars like a real dinosaur",
            "Remote control included",
            "Educational and entertaining",
        ],
    },
    {
        "title": "Building Blocks Deluxe Set",
        "short_description": "Unleash your creativity with this deluxe set of building blocks for endless fun.",
        "price": 34.99,
        "category": "Toys",
        "features": [
            "Includes 500+ colorful building blocks",
            "Promotes STEM learning and creativity",
            "Compatible with other major brick brands",
            "Comes with a durable storage container",
            "Ideal for children ages 3 and up",
        ],
    },
]

# Prepare and embed the user_history, and calculate the mean embeddings
history_texts = [create_product_text(product) for product in user_history]
history_embeddings = create_embeddings(history_texts)

# Create user interest vector
# np.mean(): 计算多个 Embedding 的平均值
# np.mean(history_embeddings, axis=0): 整个推荐算法的核心
# 把多个用户行为转换成一个 User Profile Vector
# mean_history_embeddings: User Interest Embedding
mean_history_embeddings = np.mean(history_embeddings, axis=0)

# Filter products to remove any in user_history
# 不希望推荐 user_history 中的 product
products_filtered = [product for product in products if product not in user_history]

# Combine product features and embed the resulting texts
product_texts = [create_product_text(product) for product in products_filtered]
product_embeddings = create_embeddings(product_texts)

# Find similar products
hits = find_n_closest(mean_history_embeddings, product_embeddings)

for hit in hits:
    product = products_filtered[hit["index"]]
    print(product["title"])


Robot Building Kit
Interactive Robot Pet
LEGO Space Shuttle
